In [4]:
pip install pytesseract

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from PIL import Image
import pytesseract
import spacy

# Define the backbone model
class BackboneModel(tf.keras.Model):
    def __init__(self):
        super(BackboneModel, self).__init__()
        self.conv_layers = models.Sequential([
            layers.Conv2D(32, (3, 3), strides=1, padding='same', input_shape=(128, 128, 3)),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(64, (3, 3), strides=1, padding='same'),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(128, (3, 3), strides=1, padding='same'),
            layers.BatchNormalization(),
            layers.ReLU(),
        ])
        self.pooling = layers.MaxPooling2D(pool_size=(2, 2), strides=2)

    def call(self, x):
        x = self.conv_layers(x)
        x = self.pooling(x)
        return x

# OCR function using Tesseract
def perform_ocr(image_path):
    return pytesseract.image_to_string(Image.open(image_path))

# Named Entity Recognition for extracting brand name
def extract_brand_name(text):
    nlp = spacy.load("en_core_web_sm")  # Load a pre-trained SpaCy model
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ == "ORG":  # Check for organization entities
            return ent.text
    return None

# Final unified model
class FinalModel:
    def __init__(self):
        self.backbone_model = BackboneModel()
        self.nlp = spacy.load("en_core_web_sm")  # Load SpaCy NER model

    def preprocess_image(self, image_path):
        """
        Preprocesses the input image to the required format for the backbone model.
        """
        image = Image.open(image_path).resize((128, 128))  # Resize to (128, 128)
        image_array = np.array(image) / 255.0  # Normalize pixel values
        if len(image_array.shape) == 2:  # Handle grayscale images
            image_array = np.stack([image_array] * 3, axis=-1)
        return np.expand_dims(image_array, axis=0).astype(np.float32)

    def perform_ocr(self, image_path):
        """
        Perform OCR on the input image to extract text.
        """
        return pytesseract.image_to_string(Image.open(image_path))

    def extract_brand_name(self, text):
        """
        Extract brand name from the OCR text using SpaCy NER.
        """
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ == "ORG":  # Check for organization entities
                return ent.text
        return None

    def process(self, image_path):
        """
        Unified process method to:
        1. Extract features using the backbone model.
        2. Perform OCR to extract text.
        3. Use NER to extract the brand name from the text.
        """
        # Step 1: Preprocess the image
        preprocessed_image = self.preprocess_image(image_path)

        # Step 2: Extract features using the backbone model
        self.backbone_model.build(input_shape=(None, 128, 128, 3))  # Specify input shape
        features = self.backbone_model(preprocessed_image)
        print("Extracted Features Shape:", features.shape)

        # Step 3: Perform OCR to extract text
        ocr_text = self.perform_ocr(image_path)
        print("OCR Text:", ocr_text)

        # Step 4: Extract brand name using NER
        brand_name = self.extract_brand_name(ocr_text)
        print("Extracted Brand Name:", brand_name)

        return features, ocr_text, brand_name


In [8]:
  backbone_model = BackboneModel()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
backbone_model.summary()

Model: "backbone_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential_1 (Sequential)            │ (None, 128, 128, 128)       │          94,144 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 94,144 (367.75 KB)

 Trainable params: 93,696 (366.00 KB)

 Non-trainable params: 448 (1.75 KB)

In [10]:
final_model = FinalModel()

In [14]:
data_path = '/content/drive/MyDrive/dataset'  # Update this to the folder containing your data files

# Load the data
X = np.load(os.path.join(data_path, 'X.npy'))  # Images
Y = np.load(os.path.join(data_path, 'Y.npy'))  # Labels

# Split the data into training and testing sets (70-30 split, stratified)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, stratify=Y, random_state=42
)

# Further split training data into training and validation sets (70-30 split of training data)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.3, stratify=Y_train, random_state=42
)

# Early Stopping and Learning Rate Reduction Callbacks
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5, min_lr=1e-6)

# Train the Model
history = model.fit(X_train, Y_train,epochs=900, batch_size=32,
                    callbacks=[early_stopping, reduce_lr])

Epoch 1/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.0990 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3027
Epoch 2/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.1015 - loss: 2.2997 - val_accuracy: 0.1025 - val_loss: 2.2997
Epoch 3/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 9s 58ms/step - accuracy: 0.1019 - loss: 2.2967 - val_accuracy: 0.1029 - val_loss: 2.2967
Epoch 4/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - accuracy: 0.1044 - loss: 2.2937 - val_accuracy: 0.1054 - val_loss: 2.2937
Epoch 5/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.1049 - loss: 2.2907 - val_accuracy: 0.1059 - val_loss: 2.2907
Epoch 6/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 8s 31ms/step - accuracy: 0.1073 - loss: 2.2877 - val_accuracy: 0.1083 - val_loss: 2.2877
Epoch 7/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.1078 - loss: 2.2847 - val_accuracy: 0.1088 - val_loss: 2.2847
Epoch 8/900
139/139 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.1103 - loss: 2.2817 - 

In [16]:
model_path = "brand_detection_model.h5"
backbone_model.save(model_path)
# print(f"Model saved at {model_path}")

# Download the Model
def download_model(file_path):
    from google.colab import files
    if os.path.exists(file_path):
        files.download(file_path)
    else:
        print("File not found!")